# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Priyansh-rath18/flyrank-internship-/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

## 1. Question

### Research question

Can published website content be grouped into useful performance archetypes using observed search, engagement, and content-level signals?

More specifically:

* Can content with different levels of visibility, ranking performance, and engagement be separated into interpretable groups?
* Do these groups provide a useful basis for prioritizing content review?
* How does the clustering approach compare with a simple baseline, such as ranking content by impressions or traffic?

### Decision supported

The analysis supports a **content review prioritization decision**.

It can help identify which pages may deserve attention first, such as:

* Content with high visibility but weak click performance.
* Content with strong engagement that may deserve protection.
* Content with low visibility and weak performance that may need deeper review.
* Content that could be candidates for consolidation or monitoring.

The framework is decision-support only. It does not automatically establish that rewriting, merging, or deleting a page will improve business outcomes.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

## 2. Data

### Source

The analysis uses the public FlyRank internship warehouse available through Hugging Face.

The release identifier used in the notebook is:

`FlyRank/internship-warehouse`

The notebook reads three Parquet tables:

| Table                                           | Role                                               |
| ----------------------------------------------- | -------------------------------------------------- |
| `dim_content.parquet`                           | Content attributes and publication status          |
| `fact_content_query_90d.parquet`                | Search performance aggregated over a 90-day window |
| `fact_content_daily_performance_sample.parquet` | Daily website and search performance sample        |

### Date windows

The query table contains 90-day performance fields, including:

* 90-day impressions and clicks.
* Last-30-day impressions and clicks.
* Previous-30-day impressions and clicks.
* Average search position over the 90-day period.

The daily performance table contains aggregated daily performance fields. The exact calendar dates of the available release should be reported from the dataset metadata rather than inferred from column names.

### Data preparation

The three tables were joined using `content_hash_id`.

The analysis retained content that was:

* Published.
* Not marked as deleted.

Records were excluded from the final analysis when they were not published or were marked as deleted. Missing numeric performance values after the join were filled with zero for the selected analysis fields.

### Public-safety choices

The analysis uses content-level performance aggregates and does not expose client names, private search queries, private URLs, or individual user information.


In [ ]:
import duckdb
import numpy as np
import pandas as pd

con = duckdb.connect()

rel = "hf://datasets/FlyRank/internship-warehouse"

q90 = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(impressions_90d) AS impressions_90d,
        SUM(clicks_90d) AS clicks_90d,
        SUM(impressions_last30) AS impressions_last30,
        SUM(clicks_last30) AS clicks_last30,
        SUM(impressions_prev30) AS impressions_prev30,
        SUM(clicks_prev30) AS clicks_prev30,
        COUNT(*) AS query_count,
        SUM(rare_query_count) AS rare_query_count,
        AVG(rare_impressions_share) AS rare_impressions_share_avg,
        SUM(impressions_90d * avg_position_90d)
            / NULLIF(SUM(impressions_90d), 0)
            AS avg_position_90d_w
    FROM read_parquet('{rel}/fact_content_query_90d.parquet')
    GROUP BY content_hash_id
""").df()

perf = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(ga4_sessions) AS sessions,
        SUM(ga4_pageviews) AS pageviews,
        SUM(ga4_engaged_sessions) AS engaged_sessions,
        SUM(ga4_total_engagement_sec) AS engagement_sec,
        SUM(scroll_events) AS scroll_events,
        SUM(gsc_impressions) AS gsc_impressions_month,
        SUM(gsc_clicks) AS gsc_clicks_month,
        SUM(sessions_ai) AS sessions_ai
    FROM read_parquet(
        '{rel}/fact_content_daily_performance_sample.parquet'
    )
    GROUP BY content_hash_id
""").df()

content = con.sql(f"""
    SELECT
        content_hash_id,
        content_type,
        main_intent,
        competition_level,
        word_count,
        char_count,
        backlinks,
        search_volume,
        is_published,
        is_deleted
    FROM read_parquet('{rel}/dim_content.parquet')
""").df()

df = (
    content
    .merge(q90, on="content_hash_id", how="left")
    .merge(perf, on="content_hash_id", how="left")
)

df = df[
    (df["is_published"] == True) &
    (df["is_deleted"] == False)
].copy()

print("Final content shape:", df.shape)

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

## 3. Methodology

### Analysis design

This is an unsupervised clustering study. There is no human-provided target label indicating the correct action for each content item.

### Feature engineering

The analysis uses search, engagement, content, and authority-related signals.

Examples include:

* Log-transformed impressions and clicks.
* Click-through rate.
* Weighted average search position.
* Change in clicks between the last 30 days and the previous 30 days.
* Query count and rare-query share.
* Engagement rate and average engagement time.
* Scroll events.
* Backlinks and search volume.
* Word count.

Log transformations reduce the influence of very large count values. StandardScaler puts the numeric features on comparable scales before clustering.

### Archetype labels

The six archetypes are descriptive labels assigned after clustering:

| Archetype              | Proposed action |
| ---------------------- | --------------- |
| Star Performer         | Protect         |
| Authority Pillar       | Improve         |
| Visible Underconverter | Rewrite         |
| Buried Thin Page       | Merge           |
| Hidden Engagement Gem  | Monitor         |
| Dormant                | Prune           |

These labels are analyst interpretations of cluster profiles, not ground-truth labels.

### Baseline

A simple baseline should rank content using a single observed performance signal, such as 90-day impressions. This provides a transparent comparison against the multi-feature clustering approach.

The baseline is not a predictive model. It is a simple prioritization rule.

### Validation design

The clustering results should be evaluated using:

* Silhouette score for internal cluster separation.
* Cluster size and stability across random seeds.
* Interpretability of cluster profiles.
* Agreement between the clustering recommendations and the simple baseline.

A random train/test split is not sufficient to prove that an editorial action improves future performance. That would require a time-based evaluation or an intervention experiment.

### Leakage checks

The current analysis uses observed performance measures to describe content. It does not establish future outcomes.

Potential concerns include:

* Using the same 90-day performance window for clustering and evaluation.
* Using clicks and impressions as both clustering inputs and comparison signals.
* Treating proposed actions as if they were measured outcomes.
* Aggregating daily performance without confirming that the aggregation window matches the query-performance window.

The results should therefore be presented as descriptive and directional.


In [ ]:
num_cols = [
    "impressions_90d", "clicks_90d",
    "impressions_last30", "clicks_last30",
    "impressions_prev30", "clicks_prev30",
    "query_count", "rare_query_count",
    "rare_impressions_share_avg", "avg_position_90d_w",
    "sessions", "pageviews", "engaged_sessions",
    "engagement_sec", "scroll_events",
    "gsc_impressions_month", "gsc_clicks_month",
    "sessions_ai"
]

df[num_cols] = df[num_cols].fillna(0)

df["ctr_90d"] = (
    df["clicks_90d"] /
    df["impressions_90d"].replace(0, np.nan)
).fillna(0)

df["momentum"] = (
    (df["clicks_last30"] - df["clicks_prev30"]) /
    (df["clicks_prev30"] + 1)
)

df["engagement_rate"] = (
    df["engaged_sessions"] /
    df["sessions"].replace(0, np.nan)
).fillna(0)

df["avg_engagement_sec"] = (
    df["engagement_sec"] /
    df["sessions"].replace(0, np.nan)
).fillna(0)

df["log_impressions"] = np.log1p(df["impressions_90d"])
df["log_clicks"] = np.log1p(df["clicks_90d"])
df["log_scroll_events"] = np.log1p(df["scroll_events"])
df["log_backlinks"] = np.log1p(df["backlinks"].fillna(0))
df["log_search_volume"] = np.log1p(df["search_volume"].fillna(0))

feature_cols = [
    "log_impressions",
    "log_clicks",
    "ctr_90d",
    "avg_position_90d_w",
    "momentum",
    "query_count",
    "rare_impressions_share_avg",
    "engagement_rate",
    "avg_engagement_sec",
    "log_scroll_events",
    "log_backlinks",
    "log_search_volume",
    "word_count"
]

X = (
    df[feature_cols]
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

scaler = StandardScaler()
Xs = scaler.fit_transform(X)

sil_scores = {}

for k in range(4, 8):
    labels = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    ).fit_predict(Xs)

    sil_scores[k] = silhouette_score(
        Xs,
        labels,
        sample_size=min(5000, len(Xs)),
        random_state=42
    )

print("Silhouette scores:", sil_scores)

km = KMeans(
    n_clusters=6,
    random_state=42,
    n_init=10
)

df["cluster"] = km.fit_predict(Xs)

archetype_map = {
    3: "Star Performer",
    2: "Authority Pillar",
    0: "Visible Underconverter",
    4: "Buried Thin Page",
    5: "Hidden Engagement Gem",
    1: "Dormant"
}

action_map = {
    3: "Protect",
    2: "Improve",
    0: "Rewrite",
    4: "Merge",
    5: "Monitor",
    1: "Prune"
}

df["archetype"] = df["cluster"].map(archetype_map)
df["action"] = df["cluster"].map(action_map)

print(df["cluster"].value_counts().sort_index())

In [ ]:
profile_cols = [
    "impressions_90d",
    "clicks_90d",
    "ctr_90d",
    "avg_position_90d_w",
    "momentum",
    "query_count",
    "rare_impressions_share_avg",
    "sessions",
    "engagement_rate",
    "avg_engagement_sec",
    "scroll_events",
    "backlinks",
    "search_volume",
    "word_count"
]

cluster_profile = (
    df.groupby(["cluster", "archetype", "action"])[profile_cols]
    .mean()
    .round(3)
)

display(cluster_profile)

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 4. Results

### Descriptive findings

The six-cluster solution produces the following proposed content archetypes:

1. Star Performer — Protect
2. Authority Pillar — Improve
3. Visible Underconverter — Rewrite
4. Buried Thin Page — Merge
5. Hidden Engagement Gem — Monitor
6. Dormant — Prune

The current charts show that the archetypes differ in size and in their observed relationship between search visibility and average ranking position.

The clustering output is useful for exploring content patterns. However, cluster size alone does not establish business value, and the scatter plot does not establish causation.

### Model versus baseline

A simple impressions-based ranking is used as a baseline. The same content population and the same evaluation rules must be used for both approaches.

The table below should be populated by the validation code. No performance advantage should be claimed until the values have been measured.


In [ ]:
# Baseline: rank content by 90-day impressions.
eval_df = df[
    ["content_hash_id", "impressions_90d", "archetype", "action"]
].copy()

eval_df["baseline_rank"] = (
    eval_df["impressions_90d"]
    .rank(method="first", ascending=False)
)

# Example top 10% priority group.
top_n = max(1, int(len(eval_df) * 0.10))

baseline_top = set(
    eval_df.nlargest(top_n, "impressions_90d")["content_hash_id"]
)

# Clustering-based priority:
# Protect and Improve are treated as the proposed high-value groups.
model_top = set(
    eval_df[
        eval_df["action"].isin(["Protect", "Improve"])
    ]["content_hash_id"]
)

comparison = pd.DataFrame({
    "method": [
        "Impressions baseline",
        "Archetype priority"
    ],
    "selected_items": [
        len(baseline_top),
        len(model_top)
    ],
    "mean_impressions": [
        eval_df[
            eval_df["content_hash_id"].isin(baseline_top)
        ]["impressions_90d"].mean(),
        eval_df[
            eval_df["content_hash_id"].isin(model_top)
        ]["impressions_90d"].mean()
    ]
})

display(comparison.round(2))

## 5. Limitations

*What this work cannot claim.*

## 5. Limitations

1. **No ground-truth action labels:** The six archetypes are analyst-defined interpretations of unsupervised clusters.

2. **No causal evidence:** The analysis cannot show that rewriting, merging, pruning, or protecting a page will improve its future performance.

3. **Same-window measurements:** The clustering features are based on observed performance windows. Without a separate future evaluation period, the analysis cannot establish predictive performance.

4. **Cluster sensitivity:** K-Means results can change with feature choices, scaling, the number of clusters, and random initialization.

5. **Feature dependence:** Impressions, clicks, and ranking position are related signals. Using them together may cause some performance dimensions to influence the clusters more strongly than others.

6. **Missing values:** Missing joined performance records are filled with zero. A missing record may not always mean that the true performance was zero.

7. **Aggregation effects:** Summing performance across queries and daily records may hide differences between individual queries, dates, and content types.

8. **No intervention test:** The analysis does not include a controlled content update experiment.

9. **No automatic decision authority:** The proposed actions are suggestions for human review. They should not be applied automatically without checking content quality, intent, business importance, and technical context.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 6. Ranked Recommendations

The following playbook is based on the observed archetypes and their proposed actions. The ranking is a practical review order, not a measured estimate of business impact.

### 1. Review Visible Underconverters — Rewrite

These pages show observed search visibility but may have weaker click performance relative to their exposure.

Recommended checks:

* Search intent alignment.
* Title and description clarity.
* Relevance of the opening section.
* Search-result presentation.
* Whether the page answers the main user need.

Do not rewrite every page automatically. Review representative examples first.

### 2. Protect Star Performers

These pages appear to have strong observed performance.

Recommended checks:

* Avoid unnecessary content changes.
* Monitor ranking and click trends.
* Preserve useful internal links.
* Check for technical or content regressions.

### 3. Improve Authority Pillars

These pages may have strong authority or visibility signals but still have room for improvement.

Recommended checks:

* Content completeness.
* Internal linking.
* Coverage of related user questions.
* Evidence and information quality.
* Opportunities to improve the page's main intent match.

### 4. Review Buried Thin Pages — Merge candidates

These pages should be reviewed for low visibility and limited content depth.

Recommended checks:

* Whether the page has a distinct purpose.
* Whether another page already serves the same intent.
* Whether consolidation would reduce duplication.
* Whether the page has valuable links or historical importance.

A merge should only be proposed after manual review.

### 5. Monitor Hidden Engagement Gems

These pages may have useful engagement signals despite limited observed search visibility.

Recommended checks:

* Other acquisition channels.
* Internal navigation.
* Search demand and discoverability.
* Whether the page serves a narrow but valuable audience.

Do not prune these pages based on search visibility alone.

### 6. Review Dormant Pages — Prune candidates

These pages may have limited observed activity.

Recommended checks:

* Whether the page is still needed.
* Whether it has business or informational value.
* Whether it has backlinks or historical importance.
* Whether it should be refreshed, redirected, consolidated, or retained.

Pruning is a final decision after review, not an automatic result of clustering.

### Operational recommendation

Use the archetypes to create a review queue. Start with a small sample from each group, validate the interpretation, and record the final human decision. Future work should measure whether the selected actions improve performance over a separate time window.


In [ ]:
action_playbook = (
    df[
        [
            "content_hash_id",
            "archetype",
            "action",
            "impressions_90d",
            "clicks_90d",
            "ctr_90d",
            "avg_position_90d_w",
            "momentum",
            "sessions",
            "engagement_rate"
        ]
    ]
    .sort_values(
        ["action", "impressions_90d"],
        ascending=[True, False]
    )
)

action_playbook.to_csv(
    "content_action_playbook.csv",
    index=False
)

print("Saved content_action_playbook.csv")

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## 7. Artifacts the Paper Embeds

The deployed paper should include the following artifacts:

### Figure 1 — Content archetypes by size

A bar chart showing the number of content items in each archetype.

Purpose: show how the content population is distributed across the six groups.

### Figure 2 — Visibility versus ranking position

A scatter plot showing log-transformed 90-day impressions against weighted average search position.

Purpose: show how the clusters relate to observed visibility and ranking.

### Table 1 — Dataset summary

Include:

* Source release.
* Tables used.
* Final number of published, non-deleted content items.
* Performance windows.
* Missing-value treatment.

### Table 2 — Feature definitions

Include each feature, its source field, and its meaning.

### Table 3 — Cluster profiles

Include the mean values of the major performance and content features for each archetype.

### Table 4 — Baseline comparison

Include the actual measured results of the baseline and clustering-based prioritization method using the same evaluation design.

### Table 5 — Action playbook

Include each archetype, proposed action, observed pattern, and recommended manual checks.


In [ ]:
import matplotlib.pyplot as plt

# Figure 1: cluster sizes
order = df["cluster"].value_counts().sort_values(
    ascending=False
).index

sizes = df["cluster"].value_counts().loc[order]

labels = [
    f"{archetype_map[c]} ({action_map[c]})"
    for c in order
]

plt.figure(figsize=(9, 5))
plt.bar(labels, sizes.values)
plt.ylabel("Number of content items")
plt.title("Content Archetypes by Size")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(
    "archetype_sizes.png",
    dpi=150,
    bbox_inches="tight"
)
plt.show()

# Figure 2: visibility versus ranking position
sample = df.sample(
    min(5000, len(df)),
    random_state=42
)

plt.figure(figsize=(9, 6))
plt.scatter(
    sample["log_impressions"],
    sample["avg_position_90d_w"],
    c=sample["cluster"],
    cmap="tab10",
    s=12,
    alpha=0.6
)

plt.gca().invert_yaxis()
plt.xlabel("Log(1 + impressions, 90d)")
plt.ylabel("Average search position (lower = better)")
plt.title(
    "Content Archetypes: Visibility vs Ranking Position"
)
plt.tight_layout()
plt.savefig(
    "archetype_scatter.png",
    dpi=150,
    bbox_inches="tight"
)
plt.show()

# Tables
cluster_profile.to_csv("cluster_profiles.csv")
comparison.to_csv("baseline_comparison.csv", index=False)

print("Charts and tables generated.")

## Acknowledgments & Data Credit

This work was completed as part of the FlyRank internship assignment.

The analysis uses the publicly available FlyRank internship warehouse:

https://flyrank.ai

The results are based on the available dataset release and the analysis described in this notebook. The archetypes and recommendations are analytical interpretations created for decision-support purposes.


## ML-12 — 5-Minute Demo Outline

### 0:00–0:45 — Problem

Explain that content teams may have many pages but limited time to decide which pages to review first.

Introduce the question: Can observed content performance be grouped into useful archetypes?

### 0:45–1:30 — Data

Explain the three warehouse tables, the content-level join, the 90-day search signals, and the filtering of unpublished or deleted content.

### 1:30–2:30 — Method

Show the engineered features, standardization, K-Means clustering, and the six descriptive archetypes.

Explain that the archetype names and actions were assigned after examining cluster profiles.

### 2:30–3:30 — Results

Show the archetype-size chart and visibility-versus-ranking scatter plot.

Explain what the charts show, without claiming that the clusters caused performance differences.

### 3:30–4:20 — Action playbook

Show how the archetypes map to Protect, Improve, Rewrite, Merge, Monitor, and Prune.

Explain that these are review priorities and not automatic editorial decisions.

### 4:20–5:00 — Limitations and next steps

Discuss the baseline, the lack of causal evidence, and the need for a time-based evaluation or controlled content experiment.


Built a content-archetype analysis using DuckDB, Python, and K-Means clustering.

I combined search visibility, clicks, ranking position, engagement, backlinks, and content features to group published content into six descriptive archetypes.

The output maps these groups to a practical review playbook: Protect, Improve, Rewrite, Merge, Monitor, and Prune.

The important lesson: clustering can support prioritization, but it does not prove that a content action will improve rankings or traffic.

Next step: validate the recommendations using a separate future performance window or controlled experiment.

#Python #DataScience #MachineLearning #SEO #Analytics


I built a reproducible content-analysis workflow using DuckDB, Pandas, and scikit-learn to combine multiple Parquet tables and engineer content-performance features. I used K-Means clustering to identify six descriptive content archetypes and translated their profiles into a practical content-review playbook. The project strengthened my experience in data preparation, unsupervised learning, analytical validation, and communicating model outputs as decision-support rather than overstated business claims.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
